In [1]:
#load everything(data,baseline model,validation folds,etc)
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from eventflow.baselines import CurrentRowLR, NGram, Uniform
from eventflow.config import EVENT_CLASSES, EXPERIMENT_LOG, N_CLASSES, OUTPUTS_DIR, TUNING_FOLD
from eventflow.data import encode_events, load_train, make_train_examples
from eventflow.tracking import log_experiment
from eventflow.validation import add_folds, evaluate, iter_folds, split

train = load_train()
examples = add_folds(make_train_examples(train))
print(f"{len(examples):,} practice questions from {examples['sequence_id'].nunique():,} machines")

596,953 practice questions from 6,500 machines


In [2]:
# Temporary helper until the shared cross_validate() is ready. Same idea: train on 4 folds,
# score on the 5th, repeat for every fold. It also keeps the out-of-fold predictions: for every
# practice question, the probabilities from the one model that did NOT see its machine.
def run_folds(make_model, folds=None):
    oof = np.full((len(examples), N_CLASSES), np.nan)
    scores = []
    for fold, tr, va in iter_folds(examples, folds):
        proba = make_model().fit(train, tr).predict_proba(train, va)
        oof[va.index.to_numpy()] = proba
        scores.append({"fold": fold, **evaluate(va, proba)})
    return pd.DataFrame(scores), oof

In [3]:
# Hand check: the last-event table without smoothing (alpha=0) must equal plain counting, no adding backup/short
tr, va = split(examples, TUNING_FOLD)
codes = encode_events(train["event_type"])
model = NGram(order=1, alpha=0).fit(train, tr)

question = va.iloc[[0]]                                          # one validation question
last_event = codes[question["last_row"].iloc[0]]                 # its last visible event
same_last = tr[codes[tr["last_row"].to_numpy()] == last_event]   # training questions with that last event
by_hand = np.bincount(same_last["target"], minlength=N_CLASSES) / len(same_last)

assert np.allclose(model.predict_proba(train, question)[0], by_hand)
print(f"Hand check passed. Last event {EVENT_CLASSES[last_event]}, seen {len(same_last):,} times. Most likely next:")
print(pd.Series(by_hand, index=EVENT_CLASSES).sort_values(ascending=False).head(3).round(3))

Hand check passed. Last event VIBRATION_WARNING, seen 52,882 times. Most likely next:
VIBRATION_WARNING    0.298
MAINTENANCE_CHECK    0.261
TEMP_WARNING         0.129
dtype: float64


In [4]:
# Tune the smoothing (alpha) on fold 0 only, for tables with the last 1, 2, 3 and 4 events.
# find the best alpha by cal the logloss of every alpha and order(pattern length) combiation
rows = []
for order in [1, 2, 3, 4]:
    for alpha in [0.1, 1, 10, 100, 1000]:
        scores, _ = run_folds(lambda: NGram(order, alpha), folds=[TUNING_FOLD])
        rows.append({"order": order, "alpha": alpha, "logloss": scores["logloss"].iloc[0]})
tuning = pd.DataFrame(rows).pivot(index="alpha", columns="order", values="logloss")
best_alpha = tuning.idxmin()
print("Best alpha per order:", best_alpha.to_dict())
tuning.round(4)

Best alpha per order: {1: 0.1, 2: 10.0, 3: 100.0, 4: 100.0}


order,1,2,3,4
alpha,,,,
0.1,1.7701,1.6738,1.7129,2.0043
1.0,1.7701,1.6735,1.6807,1.8086
10.0,1.7701,1.6732,1.6538,1.6788
100.0,1.7701,1.6732,1.6435,1.6379
1000.0,1.7709,1.6824,1.6597,1.6530


the alpha increases as the len of pattern increases bc for longer pattern len, higher chance of lower count

In [5]:
# All baselines on the 5 folds (the logistic regression takes a few minutes).
# Out-of-fold predictions go to outputs/oof/ for the final comparison and the error analysis.
a1, a3 = float(best_alpha[1]), float(best_alpha[3])
baselines = {                      # name: (how to make the model, settings to log)
    "uniform":         (lambda: Uniform(), {}),
    "class_prior":     (lambda: NGram(order=0, alpha=1), {"order": 0, "alpha": 1}),
    "last1_table":     (lambda: NGram(order=1, alpha=a1), {"order": 1, "alpha": a1}),
    "last1_table_all": (lambda: NGram(order=1, alpha=a1, source="all"), {"order": 1, "alpha": a1, "source": "all"}),
    "last3_table":     (lambda: NGram(order=3, alpha=a3), {"order": 3, "alpha": a3}),
    "current_row_lr":  (lambda: CurrentRowLR(), {"max_iter": 400}),
}
oof_dir = OUTPUTS_DIR / "oof"
oof_dir.mkdir(parents=True, exist_ok=True)

results = {}
for name, (make_model, settings) in baselines.items():
    scores, oof = run_folds(make_model)
    np.save(oof_dir / f"{name}.npy", oof)
    results[name] = {"logloss": scores["logloss"].mean(), "std": scores["logloss"].std(),
                     "mimic": scores["logloss_mimic"].mean()}
    print(f"{name:16s} {results[name]['logloss']:.4f} ± {results[name]['std']:.4f}")
results = pd.DataFrame(results).T
results.round(4)

uniform          2.6391 ± 0.0000
class_prior      2.1638 ± 0.0042
last1_table      1.7710 ± 0.0029
last1_table_all  1.7746 ± 0.0028
last3_table      1.6440 ± 0.0034
current_row_lr   1.7228 ± 0.0025


,logloss,std,mimic
uniform,2.6391,0.0000,2.6391
class_prior,2.1638,0.0042,2.1642
last1_table,1.7710,0.0029,1.7665
last1_table_all,1.7746,0.0028,1.7703
last3_table,1.6440,0.0034,1.6367
current_row_lr,1.7228,0.0025,1.7147


In [6]:
# Log each baseline to experiments/results.csv. Run this cell ONCE (re-running adds duplicate rows).
for name, (make_model, settings) in baselines.items():
    r = results.loc[name]
    log_experiment(run_id=f"baseline_{name}", author="Nam", model=name, feature_set="baseline",
                   val_logloss=r["logloss"],
                   params={**settings, "fold_std": round(r["std"], 5), "logloss_mimic": round(r["mimic"], 5)},
                   notes="5-fold mean, folds grouped by machine")
print(pd.read_csv(EXPERIMENT_LOG).tail(len(baselines)))

             timestamp                    run_id author            model  \
0  2026-11-28T14:08:32          baseline_uniform    Nam          uniform   
1  2026-11-28T14:08:32      baseline_class_prior    Nam      class_prior   
2  2026-11-28T14:08:32      baseline_last1_table    Nam      last1_table   
3  2026-11-28T14:08:32  baseline_last1_table_all    Nam  last1_table_all   
4  2026-11-28T14:08:32      baseline_last3_table    Nam      last3_table   
5  2026-11-28T14:08:32   baseline_current_row_lr    Nam   current_row_lr   

  feature_set  val_logloss  test_logloss  \
0    baseline      2.63906           NaN   
1    baseline      2.16384           NaN   
2    baseline      1.77102           NaN   
3    baseline      1.77462           NaN   
4    baseline      1.64404           NaN   
5    baseline      1.72278           NaN   

                                              params  \
0        {"fold_std": 0.0, "logloss_mimic": 2.63906}   
1  {"alpha": 1, "fold_std": 0.00421, "logloss_

## Notes

- **Hand check:** the last-event table without smoothing (alpha=0) equals plain counting.
- **Tuning (fold 0):** longer patterns need more smoothing (best alpha 0.1 / 10 / 100 / 100 for the last 1 / 2 / 3 / 4 events), because the same questions are spread over more patterns, so each pattern has fewer examples. For the last event alone, alpha hardly matters.
- **With smoothing, longer patterns keep helping up to 4 events:** 1.770 → 1.673 → 1.644 → 1.638 (fold 0). Without smoothing, the EDA found that 4–5 events got worse.
- **5-fold results:** uniform 2.639, class prior 2.164, last-event table 1.771, current-row LR 1.723, last-3 table **1.644** (best baseline).
- The last row's sensors help a little (the LR beats the last-event table by 0.048), but 3 events of history help more (the last-3 table beats the LR by 0.079).
- **Experiment:** learning the last-event table from all events (warm-up included) is slightly worse than learning only from test-like questions (1.7746 vs 1.7710; mimic 1.7703 vs 1.7665), which fits the warm-up behaving differently.
- LightGBM (quick check on fold 0, 1.504) beats the best baseline by 0.14.
